# Dissertation — Combined Polymarket Full-History Extraction

This notebook extracts the **maximum available YES-probability history** for all Polymarket signals used in the dissertation:

### Bitcoin vs. Gold vs. S&P 500 in 2026
- Bitcoin best performer in 2026
- Gold best performer in 2026
- S&P 500 best performer in 2026

### Will Bitcoin outperform Gold in 2025?
- YES probability

### Will Bitcoin outperform Gold in 2026?
- YES probability

The notebook dynamically discovers the market IDs and YES-token IDs, downloads history in safe chunks, automatically splits HTTP-400 windows, converts the data to daily last-observation probabilities, normalizes the three-way market, and exports everything into one Excel workbook.


In [ ]:
!pip -q install requests pandas openpyxl

import ast
import json
import math
import time
from collections import deque
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from google.colab import files


In [ ]:
# ============================================================
# STEP 2 — TARGET EVENTS
# ============================================================

GAMMA_API = "https://gamma-api.polymarket.com"
CLOB_HISTORY_URL = "https://clob.polymarket.com/prices-history"

EVENT_CONFIG = [
    {
        "event_slug": "bitcoin-vs-gold-vs-sp-500-in-2026",
        "group_id": "Best_Asset_2026",
        "signal_type": "Competing_Assets",
        "horizon_year": 2026
    },
    {
        # Displayed title:
        # "Will Bitcoin outperform Gold in 2025?"
        # Actual Polymarket event slug:
        "event_slug": "will-bitcoin-outperform-gold-in-july",
        "group_id": "BTC_Outperform_Gold_2025",
        "signal_type": "Binary_Outperformance",
        "horizon_year": 2025
    },
    {
        "event_slug": "will-bitcoin-outperform-gold-in-2026",
        "group_id": "BTC_Outperform_Gold_2026",
        "signal_type": "Binary_Outperformance",
        "horizon_year": 2026
    }
]

INITIAL_CHUNK_DAYS = 7
MINIMUM_CHUNK_HOURS = 6
FIDELITY_MINUTES = 60
PAUSE_BETWEEN_REQUESTS = 0.20

OUTPUT_FOLDER = Path("/content/polymarket_dissertation_all_markets")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("Mode: MAXIMUM AVAILABLE HISTORY")
print("Fidelity:", FIDELITY_MINUTES, "minutes")
print("Output folder:", OUTPUT_FOLDER)


In [ ]:
# ============================================================
# STEP 3 — HELPERS
# ============================================================

def safe_json_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if not isinstance(value, str):
        return []

    value = value.strip()

    if not value:
        return []

    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, list) else []
    except (json.JSONDecodeError, TypeError):
        pass

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


def parse_datetime_utc(value):
    if value is None or str(value).strip() == "":
        return pd.NaT

    return pd.to_datetime(
        value,
        utc=True,
        errors="coerce"
    )


def get_json(url, params=None, max_retries=6, timeout=90):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )

            if response.status_code == 429:
                wait_seconds = min(attempt * 10, 60)
                print(f"Rate limited. Waiting {wait_seconds}s...")
                time.sleep(wait_seconds)
                continue

            if response.status_code in {500, 502, 503, 504}:
                wait_seconds = min(attempt * 5, 30)
                print(
                    f"Temporary API error {response.status_code}. "
                    f"Waiting {wait_seconds}s..."
                )
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            return response.json()

        except requests.RequestException as error:
            last_error = error

            if attempt == max_retries:
                break

            time.sleep(min(attempt * 5, 30))

    raise RuntimeError(
        f"Request failed after retries: {last_error}"
    )


## STEP 4 — Resolve all target markets

The notebook does not hard-code token IDs. It retrieves each event from Gamma and identifies the YES CLOB token automatically.


In [ ]:
# ============================================================
# STEP 4 — FETCH EVENTS + BUILD MARKET TABLE
# ============================================================

event_rows = []
market_rows = []

for cfg in EVENT_CONFIG:

    event_url = (
        f"{GAMMA_API}/events/slug/{cfg['event_slug']}"
    )

    print("\nFetching event:", cfg["event_slug"])

    event = get_json(event_url)

    event_rows.append({
        "Event_ID": str(event.get("id")),
        "Event_Slug": event.get("slug"),
        "Event_Title": event.get("title"),
        "Group_ID": cfg["group_id"],
        "Signal_Type": cfg["signal_type"],
        "Horizon_Year": cfg["horizon_year"],
        "Event_Start_UTC": event.get("startDate"),
        "Event_Creation_UTC": (
            event.get("creationDate")
            or event.get("createdAt")
        ),
        "Event_End_UTC": event.get("endDate"),
        "Event_Active": event.get("active"),
        "Event_Closed": event.get("closed")
    })

    markets = event.get("markets", [])

    if not markets:
        raise ValueError(
            f"No child markets returned for {cfg['event_slug']}"
        )

    for market in markets:

        outcomes = safe_json_list(
            market.get("outcomes")
        )

        token_ids = safe_json_list(
            market.get("clobTokenIds")
        )

        if len(outcomes) != len(token_ids):
            continue

        yes_positions = [
            i
            for i, outcome in enumerate(outcomes)
            if str(outcome).strip().casefold() == "yes"
        ]

        if len(yes_positions) != 1:
            continue

        question = str(
            market.get("question", "")
        ).strip()

        group_title = str(
            market.get("groupItemTitle", "")
        ).strip()

        include_market = False
        series_name = None
        asset = None

        if cfg["group_id"] == "Best_Asset_2026":

            if group_title in {
                "Bitcoin",
                "Gold",
                "S&P 500"
            }:
                include_market = True
                asset = group_title

                if group_title == "Bitcoin":
                    series_name = "Bitcoin_Best_2026"

                elif group_title == "Gold":
                    series_name = "Gold_Best_2026"

                elif group_title == "S&P 500":
                    series_name = "SP500_Best_2026"

        elif cfg["group_id"] == "BTC_Outperform_Gold_2025":

            if (
                "bitcoin outperform gold" in question.casefold()
                and "2025" in question
            ):
                include_market = True
                asset = "Bitcoin vs Gold"
                series_name = "BTC_Outperform_Gold_2025"

        elif cfg["group_id"] == "BTC_Outperform_Gold_2026":

            if (
                "bitcoin outperform gold" in question.casefold()
                and "2026" in question
            ):
                include_market = True
                asset = "Bitcoin vs Gold"
                series_name = "BTC_Outperform_Gold_2026"

        if not include_market:
            continue

        yes_index = yes_positions[0]

        market_rows.append({
            "Event_ID": str(event.get("id")),
            "Event_Slug": event.get("slug"),
            "Event_Title": event.get("title"),
            "Group_ID": cfg["group_id"],
            "Signal_Type": cfg["signal_type"],
            "Horizon_Year": cfg["horizon_year"],
            "Series": series_name,
            "Asset": asset,
            "Market_ID": str(market.get("id")),
            "Market_Slug": market.get("slug"),
            "Question": question,
            "Condition_ID": market.get("conditionId"),
            "YES_Token_ID": str(token_ids[yes_index]),
            "Market_Created_UTC": (
                market.get("createdAt")
                or market.get("creationDate")
            ),
            "Market_Start_UTC": market.get("startDate"),
            "Accepting_Orders_UTC": market.get(
                "acceptingOrdersTimestamp"
            ),
            "Market_End_UTC": market.get("endDate"),
            "Market_Closed_Time_UTC": market.get("closedTime"),
            "Market_Updated_UTC": market.get("updatedAt"),
            "Active": market.get("active"),
            "Closed": market.get("closed")
        })


event_info_df = pd.DataFrame(event_rows)
market_info_df = pd.DataFrame(market_rows)


EXPECTED_SERIES = {
    "Bitcoin_Best_2026",
    "Gold_Best_2026",
    "SP500_Best_2026",
    "BTC_Outperform_Gold_2025",
    "BTC_Outperform_Gold_2026"
}

found_series = set(market_info_df["Series"])
missing_series = EXPECTED_SERIES - found_series

if missing_series:
    raise ValueError(
        f"Missing dissertation markets: {missing_series}"
    )


series_order = {
    "Bitcoin_Best_2026": 1,
    "Gold_Best_2026": 2,
    "SP500_Best_2026": 3,
    "BTC_Outperform_Gold_2025": 4,
    "BTC_Outperform_Gold_2026": 5
}

market_info_df["Order"] = (
    market_info_df["Series"].map(series_order)
)

market_info_df = (
    market_info_df
    .sort_values("Order")
    .drop(columns="Order")
    .reset_index(drop=True)
)

display(
    market_info_df[
        [
            "Group_ID",
            "Horizon_Year",
            "Series",
            "Market_ID",
            "Question",
            "YES_Token_ID",
            "Active",
            "Closed"
        ]
    ]
)


## STEP 5 — Determine the maximum extraction period

Active markets run through the current UTC time. Closed markets stop around their actual closure/end boundary so the notebook does not waste requests on long empty post-resolution periods.


In [ ]:
# ============================================================
# STEP 5 — HISTORY BOUNDARIES
# ============================================================

now_utc = pd.Timestamp.now(tz="UTC").floor("s")


def choose_history_start(row):

    accepting = parse_datetime_utc(
        row["Accepting_Orders_UTC"]
    )

    if not pd.isna(accepting):
        return pd.to_datetime(
            math.floor(accepting.timestamp()),
            unit="s",
            utc=True
        )

    market_start = parse_datetime_utc(
        row["Market_Start_UTC"]
    )

    if not pd.isna(market_start):
        return pd.to_datetime(
            math.floor(market_start.timestamp()),
            unit="s",
            utc=True
        )

    created = parse_datetime_utc(
        row["Market_Created_UTC"]
    )

    if not pd.isna(created):
        return pd.to_datetime(
            math.floor(created.timestamp()),
            unit="s",
            utc=True
        )

    raise ValueError(
        f"No usable start timestamp for {row['Series']}"
    )


def choose_history_end(row):

    is_closed = (
        row["Closed"] is True
        or str(row["Closed"]).lower() == "true"
    )

    if not is_closed:
        return now_utc

    closed_time = parse_datetime_utc(
        row["Market_Closed_Time_UTC"]
    )

    if not pd.isna(closed_time):
        return min(
            closed_time + pd.Timedelta(hours=6),
            now_utc
        )

    end_date = parse_datetime_utc(
        row["Market_End_UTC"]
    )

    if not pd.isna(end_date):
        return min(
            end_date + pd.Timedelta(days=7),
            now_utc
        )

    return now_utc


market_info_df["Requested_Start_UTC"] = (
    market_info_df.apply(
        choose_history_start,
        axis=1
    )
)

market_info_df["Requested_End_UTC"] = (
    market_info_df.apply(
        choose_history_end,
        axis=1
    )
)

display(
    market_info_df[
        [
            "Series",
            "Requested_Start_UTC",
            "Requested_End_UTC",
            "Active",
            "Closed"
        ]
    ]
)


## STEP 6 — Robust CLOB history downloader

The API is queried in 7-day windows. If a window returns HTTP 400, it is automatically split into smaller windows.


In [ ]:
# ============================================================
# STEP 6.1 — ONE CLOB REQUEST
# ============================================================

def request_clob_history_window(
    token_id,
    start_utc,
    end_utc,
    fidelity_minutes=60,
    max_retries=5,
    timeout=90
):
    params = {
        "market": str(token_id),
        "startTs": int(start_utc.timestamp()),
        "endTs": int(end_utc.timestamp()),
        "fidelity": int(fidelity_minutes)
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                CLOB_HISTORY_URL,
                params=params,
                headers=headers,
                timeout=timeout
            )

            if response.status_code == 200:

                try:
                    payload = response.json()
                except ValueError:
                    return {
                        "success": False,
                        "status_code": 200,
                        "history": [],
                        "error": "Invalid JSON response",
                        "request_url": response.url
                    }

                return {
                    "success": True,
                    "status_code": 200,
                    "history": payload.get("history", []) or [],
                    "error": None,
                    "request_url": response.url
                }

            if response.status_code == 400:

                try:
                    error_message = response.json()
                except ValueError:
                    error_message = response.text

                return {
                    "success": False,
                    "status_code": 400,
                    "history": [],
                    "error": error_message,
                    "request_url": response.url
                }

            if response.status_code == 429:

                wait_seconds = min(attempt * 10, 60)

                print(
                    f"    Rate limited. "
                    f"Waiting {wait_seconds}s..."
                )

                time.sleep(wait_seconds)
                continue

            if response.status_code in {500, 502, 503, 504}:

                wait_seconds = min(attempt * 5, 30)

                print(
                    f"    Server error "
                    f"{response.status_code}. "
                    f"Waiting {wait_seconds}s..."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

        except requests.RequestException as error:

            if attempt == max_retries:

                return {
                    "success": False,
                    "status_code": None,
                    "history": [],
                    "error": str(error),
                    "request_url": None
                }

            time.sleep(min(attempt * 5, 30))

    return {
        "success": False,
        "status_code": None,
        "history": [],
        "error": "Maximum retries reached",
        "request_url": None
    }


def history_payload_to_dataframe(history):

    if not history:
        return pd.DataFrame()

    frame = pd.DataFrame(history)

    if not {"t", "p"}.issubset(frame.columns):
        return pd.DataFrame()

    frame = frame.rename(
        columns={
            "t": "Timestamp_Unix",
            "p": "Probability"
        }
    )

    frame["Timestamp_Unix"] = pd.to_numeric(
        frame["Timestamp_Unix"],
        errors="coerce"
    )

    frame["Probability"] = pd.to_numeric(
        frame["Probability"],
        errors="coerce"
    )

    return frame.dropna(
        subset=[
            "Timestamp_Unix",
            "Probability"
        ]
    )


In [ ]:
# ============================================================
# STEP 6.2 — WINDOW CREATION + AUTO-SPLITTING
# ============================================================

def create_initial_windows(
    start_utc,
    end_utc,
    chunk_days
):
    windows = []
    current_start = start_utc

    while current_start < end_utc:

        current_end = min(
            current_start
            + pd.Timedelta(days=chunk_days),
            end_utc
        )

        windows.append(
            (current_start, current_end)
        )

        current_start = current_end

    return windows


def fetch_full_token_history(
    token_id,
    start_utc,
    end_utc,
    initial_chunk_days=7,
    minimum_chunk_hours=6,
    fidelity_minutes=60
):

    queue = deque(
        create_initial_windows(
            start_utc,
            end_utc,
            initial_chunk_days
        )
    )

    successful_frames = []
    failed_rows = []
    request_counter = 0

    while queue:

        window_start, window_end = queue.popleft()

        request_counter += 1

        window_hours = (
            window_end - window_start
        ).total_seconds() / 3600

        print(
            f"  Request {request_counter}: "
            f"{window_start} -> {window_end}"
        )

        result = request_clob_history_window(
            token_id=token_id,
            start_utc=window_start,
            end_utc=window_end,
            fidelity_minutes=fidelity_minutes
        )

        if result["success"]:

            chunk_df = history_payload_to_dataframe(
                result["history"]
            )

            if not chunk_df.empty:

                successful_frames.append(chunk_df)

                print(
                    f"    Success: "
                    f"{len(chunk_df)} observations"
                )

            else:
                print("    Success, no observations.")

            time.sleep(PAUSE_BETWEEN_REQUESTS)
            continue

        if (
            result["status_code"] == 400
            and window_hours > minimum_chunk_hours
        ):

            midpoint = (
                window_start
                + (window_end - window_start) / 2
            )

            queue.appendleft(
                (midpoint, window_end)
            )

            queue.appendleft(
                (window_start, midpoint)
            )

            print(
                "    HTTP 400 -> splitting window."
            )

            continue

        print(
            "    Permanent failure:",
            result["error"]
        )

        failed_rows.append({
            "Token_ID": str(token_id),
            "Window_Start_UTC": window_start,
            "Window_End_UTC": window_end,
            "Window_Hours": window_hours,
            "Status_Code": result["status_code"],
            "Error": str(result["error"]),
            "Request_URL": result["request_url"]
        })

    if not successful_frames:
        return (
            pd.DataFrame(),
            pd.DataFrame(failed_rows)
        )

    history_df = pd.concat(
        successful_frames,
        ignore_index=True
    )

    history_df["Datetime_UTC"] = pd.to_datetime(
        history_df["Timestamp_Unix"],
        unit="s",
        utc=True,
        errors="coerce"
    )

    history_df = (
        history_df
        .dropna(
            subset=[
                "Datetime_UTC",
                "Probability"
            ]
        )
        .loc[
            lambda x:
            x["Probability"].between(
                0,
                1,
                inclusive="both"
            )
        ]
        .sort_values("Datetime_UTC")
        .drop_duplicates(
            subset=["Timestamp_Unix"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    history_df["Date"] = (
        history_df["Datetime_UTC"]
        .dt.floor("D")
        .dt.tz_localize(None)
    )

    return (
        history_df,
        pd.DataFrame(failed_rows)
    )


## STEP 7 — Download all five probability series


In [ ]:
# ============================================================
# STEP 7 — DOWNLOAD
# ============================================================

all_history = []
all_failures = []
download_log = []

for _, row in market_info_df.iterrows():

    print("\n" + "=" * 100)
    print("Downloading:", row["Series"])
    print("Question:", row["Question"])
    print("Market ID:", row["Market_ID"])
    print(
        "Range:",
        row["Requested_Start_UTC"],
        "to",
        row["Requested_End_UTC"]
    )

    history_df, failures_df = fetch_full_token_history(
        token_id=row["YES_Token_ID"],
        start_utc=row["Requested_Start_UTC"],
        end_utc=row["Requested_End_UTC"],
        initial_chunk_days=INITIAL_CHUNK_DAYS,
        minimum_chunk_hours=MINIMUM_CHUNK_HOURS,
        fidelity_minutes=FIDELITY_MINUTES
    )

    if not failures_df.empty:

        failures_df["Series"] = row["Series"]
        failures_df["Group_ID"] = row["Group_ID"]
        failures_df["Market_ID"] = row["Market_ID"]

        all_failures.append(failures_df)

    if history_df.empty:

        download_log.append({
            "Group_ID": row["Group_ID"],
            "Series": row["Series"],
            "Market_ID": row["Market_ID"],
            "Status": "No data returned",
            "Requested_Start_UTC": row["Requested_Start_UTC"],
            "Requested_End_UTC": row["Requested_End_UTC"],
            "First_Returned_UTC": None,
            "Last_Returned_UTC": None,
            "Raw_Observations": 0,
            "Available_Days": 0
        })

        continue

    history_df["Event_ID"] = row["Event_ID"]
    history_df["Event_Slug"] = row["Event_Slug"]
    history_df["Event_Title"] = row["Event_Title"]
    history_df["Group_ID"] = row["Group_ID"]
    history_df["Signal_Type"] = row["Signal_Type"]
    history_df["Horizon_Year"] = row["Horizon_Year"]
    history_df["Series"] = row["Series"]
    history_df["Asset"] = row["Asset"]
    history_df["Market_ID"] = row["Market_ID"]
    history_df["YES_Token_ID"] = row["YES_Token_ID"]

    all_history.append(history_df)

    download_log.append({
        "Group_ID": row["Group_ID"],
        "Series": row["Series"],
        "Market_ID": row["Market_ID"],
        "Status": "Success",
        "Requested_Start_UTC": row["Requested_Start_UTC"],
        "Requested_End_UTC": row["Requested_End_UTC"],
        "First_Returned_UTC": history_df["Datetime_UTC"].min(),
        "Last_Returned_UTC": history_df["Datetime_UTC"].max(),
        "Raw_Observations": len(history_df),
        "Available_Days": history_df["Date"].nunique()
    })


if not all_history:
    raise ValueError(
        "No CLOB price history was returned."
    )


history_long = pd.concat(
    all_history,
    ignore_index=True
)

history_long = (
    history_long
    .sort_values(
        [
            "Datetime_UTC",
            "Group_ID",
            "Series"
        ]
    )
    .reset_index(drop=True)
)

download_log_df = pd.DataFrame(download_log)

if all_failures:
    failed_windows_df = pd.concat(
        all_failures,
        ignore_index=True
    )
else:
    failed_windows_df = pd.DataFrame()

display(download_log_df)


## STEP 8 — Daily last-observation data


In [ ]:
# ============================================================
# STEP 8 — DAILY LONG
# ============================================================

daily_long = (
    history_long
    .sort_values("Datetime_UTC")
    .groupby(
        [
            "Date",
            "Event_ID",
            "Event_Slug",
            "Event_Title",
            "Group_ID",
            "Signal_Type",
            "Horizon_Year",
            "Series",
            "Asset",
            "Market_ID",
            "YES_Token_ID"
        ],
        as_index=False
    )
    .last()
)

daily_long = daily_long[
    [
        "Date",
        "Event_ID",
        "Event_Slug",
        "Event_Title",
        "Group_ID",
        "Signal_Type",
        "Horizon_Year",
        "Series",
        "Asset",
        "Market_ID",
        "YES_Token_ID",
        "Datetime_UTC",
        "Probability"
    ]
]

display(daily_long.head(20))


## STEP 9 — Combined daily-wide dataset

All five signals are retained as separate columns. Missing values are deliberately left missing because the markets have different launch dates and horizons.


In [ ]:
# ============================================================
# STEP 9 — DAILY WIDE
# ============================================================

daily_wide = (
    daily_long
    .pivot(
        index="Date",
        columns="Series",
        values="Probability"
    )
    .reset_index()
)

daily_wide.columns.name = None

daily_wide = (
    daily_wide
    .sort_values("Date")
    .reset_index(drop=True)
)

daily_wide = daily_wide.rename(
    columns={
        "Bitcoin_Best_2026":
            "Bitcoin_Best_2026_Prob",
        "Gold_Best_2026":
            "Gold_Best_2026_Prob",
        "SP500_Best_2026":
            "SP500_Best_2026_Prob",
        "BTC_Outperform_Gold_2025":
            "BTC_Outperform_Gold_2025_Prob",
        "BTC_Outperform_Gold_2026":
            "BTC_Outperform_Gold_2026_Prob"
    }
)

display(daily_wide.head(20))


## STEP 10 — Normalize the three-way 2026 event

Only Bitcoin/Gold/S&P 500 best-performer probabilities are normalized to sum to 1. The two binary outperform markets remain as their original YES probabilities.


In [ ]:
# ============================================================
# STEP 10 — THREE-WAY NORMALISATION
# ============================================================

three_way_cols = [
    "Bitcoin_Best_2026_Prob",
    "Gold_Best_2026_Prob",
    "SP500_Best_2026_Prob"
]

daily_wide["Best_Asset_2026_Prob_Sum"] = (
    daily_wide[three_way_cols]
    .sum(axis=1, min_count=3)
)

daily_wide["Bitcoin_Best_2026_Prob_Norm"] = (
    daily_wide["Bitcoin_Best_2026_Prob"]
    / daily_wide["Best_Asset_2026_Prob_Sum"]
)

daily_wide["Gold_Best_2026_Prob_Norm"] = (
    daily_wide["Gold_Best_2026_Prob"]
    / daily_wide["Best_Asset_2026_Prob_Sum"]
)

daily_wide["SP500_Best_2026_Prob_Norm"] = (
    daily_wide["SP500_Best_2026_Prob"]
    / daily_wide["Best_Asset_2026_Prob_Sum"]
)

daily_wide["Best_Asset_2026_Normalized_Sum"] = (
    daily_wide[
        [
            "Bitcoin_Best_2026_Prob_Norm",
            "Gold_Best_2026_Prob_Norm",
            "SP500_Best_2026_Prob_Norm"
        ]
    ]
    .sum(axis=1, min_count=3)
)

display(daily_wide.tail(20))


## STEP 11 — Separate analysis-ready datasets


In [ ]:
# ============================================================
# STEP 11 — GROUP-SPECIFIC DAILY DATASETS
# ============================================================

best_asset_2026_daily = (
    daily_wide[
        [
            "Date",
            "Bitcoin_Best_2026_Prob",
            "Gold_Best_2026_Prob",
            "SP500_Best_2026_Prob",
            "Best_Asset_2026_Prob_Sum",
            "Bitcoin_Best_2026_Prob_Norm",
            "Gold_Best_2026_Prob_Norm",
            "SP500_Best_2026_Prob_Norm",
            "Best_Asset_2026_Normalized_Sum"
        ]
    ]
    .dropna(
        subset=[
            "Bitcoin_Best_2026_Prob",
            "Gold_Best_2026_Prob",
            "SP500_Best_2026_Prob"
        ]
    )
    .reset_index(drop=True)
)


btc_gold_2025_daily = (
    daily_wide[
        [
            "Date",
            "BTC_Outperform_Gold_2025_Prob"
        ]
    ]
    .dropna(
        subset=[
            "BTC_Outperform_Gold_2025_Prob"
        ]
    )
    .reset_index(drop=True)
)


btc_gold_2026_daily = (
    daily_wide[
        [
            "Date",
            "BTC_Outperform_Gold_2026_Prob"
        ]
    ]
    .dropna(
        subset=[
            "BTC_Outperform_Gold_2026_Prob"
        ]
    )
    .reset_index(drop=True)
)


print("Best Asset 2026 days:", len(best_asset_2026_daily))
print("BTC > Gold 2025 days:", len(btc_gold_2025_daily))
print("BTC > Gold 2026 days:", len(btc_gold_2026_daily))

display(best_asset_2026_daily.head())
display(btc_gold_2025_daily.head())
display(btc_gold_2026_daily.head())


## STEP 12 — Availability / quality summary


In [ ]:
# ============================================================
# STEP 12 — AVAILABILITY SUMMARY
# ============================================================

availability_summary = (
    daily_long
    .groupby(
        [
            "Group_ID",
            "Horizon_Year",
            "Series"
        ],
        as_index=False
    )
    .agg(
        First_Date=("Date", "min"),
        Last_Date=("Date", "max"),
        Available_Days=("Date", "nunique"),
        Min_Probability=("Probability", "min"),
        Max_Probability=("Probability", "max")
    )
)

display(availability_summary)


## STEP 13 — Export all results

The Excel workbook contains metadata, raw hourly history, combined daily data, and separate sheets for each dissertation market group.


In [ ]:
# ============================================================
# STEP 13 — SAVE CSV + EXCEL OUTPUTS
# ============================================================

event_info_df.to_csv(
    OUTPUT_FOLDER / "01_event_info.csv",
    index=False
)

market_info_df.to_csv(
    OUTPUT_FOLDER / "02_market_info.csv",
    index=False
)

history_long.to_csv(
    OUTPUT_FOLDER / "03_all_hourly_history_long.csv",
    index=False
)

daily_long.to_csv(
    OUTPUT_FOLDER / "04_all_daily_long.csv",
    index=False
)

daily_wide.to_csv(
    OUTPUT_FOLDER / "05_all_daily_wide.csv",
    index=False
)

best_asset_2026_daily.to_csv(
    OUTPUT_FOLDER / "06_best_asset_2026_daily.csv",
    index=False
)

btc_gold_2025_daily.to_csv(
    OUTPUT_FOLDER / "07_btc_outperform_gold_2025_daily.csv",
    index=False
)

btc_gold_2026_daily.to_csv(
    OUTPUT_FOLDER / "08_btc_outperform_gold_2026_daily.csv",
    index=False
)

download_log_df.to_csv(
    OUTPUT_FOLDER / "09_download_log.csv",
    index=False
)

failed_windows_df.to_csv(
    OUTPUT_FOLDER / "10_failed_windows.csv",
    index=False
)


def excel_safe(df):
    df = df.copy()

    for col in df.columns:

        if isinstance(
            df[col].dtype,
            pd.DatetimeTZDtype
        ):
            df[col] = (
                df[col]
                .dt.tz_convert("UTC")
                .dt.tz_localize(None)
            )

    return df


excel_path = (
    OUTPUT_FOLDER
    / "Polymarket_Dissertation_All_Markets_FULL_History.xlsx"
)


with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    excel_safe(event_info_df).to_excel(
        writer,
        sheet_name="Event Info",
        index=False
    )

    excel_safe(market_info_df).to_excel(
        writer,
        sheet_name="Market Info",
        index=False
    )

    excel_safe(download_log_df).to_excel(
        writer,
        sheet_name="Download Log",
        index=False
    )

    excel_safe(failed_windows_df).to_excel(
        writer,
        sheet_name="Failed Windows",
        index=False
    )

    excel_safe(availability_summary).to_excel(
        writer,
        sheet_name="Availability",
        index=False
    )

    excel_safe(history_long).to_excel(
        writer,
        sheet_name="Hourly Long",
        index=False
    )

    excel_safe(daily_long).to_excel(
        writer,
        sheet_name="Daily Long",
        index=False
    )

    excel_safe(daily_wide).to_excel(
        writer,
        sheet_name="All Daily Wide",
        index=False
    )

    excel_safe(best_asset_2026_daily).to_excel(
        writer,
        sheet_name="Best Asset 2026",
        index=False
    )

    excel_safe(btc_gold_2025_daily).to_excel(
        writer,
        sheet_name="BTC Gold 2025",
        index=False
    )

    excel_safe(btc_gold_2026_daily).to_excel(
        writer,
        sheet_name="BTC Gold 2026",
        index=False
    )


print("Saved workbook:")
print(excel_path)


## STEP 14 — Download the combined workbook


In [ ]:
files.download(str(excel_path))
